# GPflow syntax: SVGP
This is the model which the SVWP model heavily relies on.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys

In [ ]:
import matplotlib
import numpy as np
import tensorflow as tf

In [ ]:
import gpflow
from gpflow.ci_utils import ci_niter
from gpflow.utilities import print_summary

In [ ]:
sys.path.append('../../FCEst-benchmarking')

from helpers.synthetic_covariance_structures import get_constant_covariances, get_periodic_covariances, get_stepwise_covariances
from helpers.synthetic_covariance_structures import get_d2_covariance_structure

In [ ]:
%matplotlib inline
matplotlib.rcParams['figure.figsize'] = (12, 6)
plt = matplotlib.pyplot

## Generate bivariate data

In [ ]:
N = 400

In [ ]:
cov_structure = get_d2_covariance_structure(
    get_periodic_covariances(n_samples=N, n_periods=1)
)
cov_structure.shape

In [ ]:
x = np.linspace(0, 1, N).reshape(-1, 1)
x.shape

In [ ]:
ts = cov_structure[:, 0, 1].reshape(-1, 1)
ts.shape

In [ ]:
plt.figure()
plt.plot(x, ts, 'x-')
plt.xlabel('time [steps]')

# Sparse Variational Gaussian Process Regression
This is an example to see the syntax of GPflow.

In [ ]:
k = gpflow.kernels.Matern52()
k

In [ ]:
M = 20  # Number of inducing locations

# Z = x[:M, :].copy()  # Initialize inducing locations to the first M inputs in the dataset
Z = np.random.choice(x[:, 0], M).reshape(-1, 1)

m = gpflow.models.SVGP(
    kernel=k, 
    likelihood=gpflow.likelihoods.Gaussian(), 
    inducing_variable=Z, 
    num_data=N
)

In [ ]:
m.likelihood.variance.assign(0.01)
m.kernel.lengthscales.assign(0.3)

In [ ]:
# predict mean and variance of latent GP at test points
mean, var = m.predict_f(x)
# mean, var = m.predict_y(x)
mean.shape

In [ ]:
plt.figure()
plt.plot(x, ts, 'kx', label="Training points")
plt.plot(x, mean, "C0", lw=2, label="Mean of predictive posterior")
plt.fill_between(
    x[:, 0],
    mean[:, 0] - 1.96 * np.sqrt(var[:, 0]),
    mean[:, 0] + 1.96 * np.sqrt(var[:, 0]),
    color="C0",
    alpha=0.2,
)
plt.plot(
    m.inducing_variable.Z.numpy(), np.zeros_like(Z), "k|",
    mew=2,
    label="Inducing locations"
)
plt.xlabel('time [steps]')
plt.legend(loc="lower right")

In [ ]:
m.inducing_variable.Z

In [ ]:
m.likelihood.variance.numpy()

## Optimization

In [ ]:
data = (x, ts)

minibatch_size = 10

train_dataset = tf.data.Dataset.from_tensor_slices((x, ts)).repeat().shuffle(N)

In [ ]:
m.log_prior_density()

In [ ]:
m.log_posterior_density(data)

In [ ]:
m.elbo(data)

In [ ]:
n_iterations = 5000
log_interval = 10

In [ ]:
posterior = m.posterior()
posterior

In [ ]:
def run_adam(model, iterations):
    """
    Utility function running the Adam optimizer
    :param model: GPflow model
    :param interations: number of iterations
    """
    # Create an Adam Optimizer action
    logf = []
    train_iter = iter(train_dataset.batch(minibatch_size))
    training_loss = model.training_loss_closure(train_iter, compile=True)
    optimizer = tf.optimizers.Adam()

    @tf.function
    def optimization_step():
        optimizer.minimize(training_loss, model.trainable_variables)

    for step in range(iterations):
        optimization_step()
        if step % log_interval == 0:
            elbo = -training_loss().numpy()
            logf.append(elbo)
    return logf

In [ ]:
# @tf.function
# def optimization_step(optimizer, model: gpflow.models.SVGP, batch):
#     with tf.GradientTape(watch_accessed_variables=False) as tape:
#         tape.watch(model.trainable_variables)
#         objective = -model.elbo(batch)
#         grads = tape.gradient(objective, model.trainable_variables)
#     optimizer.apply_gradients(zip(grads, model.trainable_variables))
#     return objective

In [ ]:
# note that you cannot continue an optimisation routine, Tensorflow cannot create variables on non-first calls
maxiter = ci_niter(n_iterations)
logf = run_adam(m, maxiter)

In [ ]:
plt.plot(np.arange(maxiter)[::log_interval], logf)
plt.xlabel("iteration")
_ = plt.ylabel("ELBO")

In [ ]:
m

In [ ]:
m.log_prior_density()

In [ ]:
m.elbo(data)

In [ ]:
m.log_posterior_density(data)

In [ ]:
# predict mean and variance of latent GP at test points
mean, var = m.predict_f(x)
mean.shape

In [ ]:
plt.figure()
plt.plot(x, ts, 'kx', label="Training points")
plt.plot(x, mean, "C0", lw=2, label="Mean of predictive posterior")
plt.fill_between(
    x[:, 0],
    mean[:, 0] - 1.96 * np.sqrt(var[:, 0]),
    mean[:, 0] + 1.96 * np.sqrt(var[:, 0]),
    color="C0",
    alpha=0.2,
)
plt.plot(
    m.inducing_variable.Z.numpy(), np.zeros_like(Z), "k|",
    mew=2,
    label="Inducing locations"
)
plt.xlabel('time [steps]')
plt.legend(loc="lower right")